In [3]:
from beir.datasets.data_loader import GenericDataLoader
from helpers.converter import convert_corpus_jsonl
import os

/Users/dieudonne/Developer/dense-sparse/.venv/lib/python3.13/site-packages/beir/datasets/data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [1]:
## necessary directories
data_path = "./datasets/nfcorpus/" ## For the normal docs
index_path = "datasets/index/" ## For the inverted index
output_doc_path = "./datasets/docs" ## for the converted docs 
qrel_file = "./datasets/nfcorpus/qrels/test.tsv"
splade_index_path = "datasets/splade-index"
splade_model = 'naver/splade-cocondenser-ensembledistil' 
splade_path = "./datasets/docs/splade"


In [4]:
## Load the dataset
corpus, queries, qrels = GenericDataLoader(data_path).load()

100%|██████████| 3633/3633 [00:00<00:00, 125051.55it/s]


In [5]:
## Convert the dataset to jsonl
new_data_path = convert_corpus_jsonl(corpus=corpus, output_path=output_doc_path)

Saved 3633 documents to ./datasets/docs/corpus.jsonl


In [5]:
from sparse.bm25 import BM25Retrieval
from sparse.splade import Splade

# Create the index folder if not exist
os.makedirs(index_path, exist_ok=True)

bm25 = BM25Retrieval(index_path) ## We create the index it does not exist
bm25.index(new_data_path) ## Index the data

splade = Splade(splade_index_path, splade_model)

apr 17, 2026 11:00:45 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFORMAZIONI: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false
Loading weights: 100%|██████████| 204/204 [00:00<00:00, 33751.65it/s]
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
## Evaluate bm25
results = {}

for qid, query in queries.items():
    hits = bm25.searcher.search(query, k=10)
    
    results[qid] = {}
    for hit in hits:
        results[qid][hit.docid] = hit.score
    
    

In [7]:
## Tokenise and encode the splade model
tokens = splade.convert_corpus(f"{new_data_path}/corpus.jsonl", "splade_vectors.jsonl")

In [14]:
## Create the inverted index for splade
splade.index(f"{new_data_path}/splade")

Loading weights: 100%|██████████| 204/204 [00:00<00:00, 25996.96it/s]
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
splade_results = {}
for qid, query in queries.items():
    hits = splade.search(query)  # use your existing method
    splade_results[qid] = {}
    for hit in hits:
        splade_results[qid][hit["docid"]] = hit["score"]

In [17]:
splade_results

{'PLAIN-2': {'MED-2429': 701.0,
  'MED-2431': 675.0,
  'MED-4827': 574.0,
  'MED-10': 570.0,
  'MED-14': 570.0,
  'MED-1732': 552.0,
  'MED-2427': 551.0,
  'MED-2440': 551.0,
  'MED-2434': 528.0,
  'MED-2122': 523.0},
 'PLAIN-12': {'MED-4711': 431.0,
  'MED-2603': 309.0,
  'MED-2501': 307.0,
  'MED-4113': 263.0,
  'MED-3317': 255.0,
  'MED-1116': 248.0,
  'MED-2028': 248.0,
  'MED-3706': 248.0,
  'MED-4114': 248.0,
  'MED-4115': 248.0},
 'PLAIN-23': {'MED-1961': 436.0,
  'MED-1169': 421.0,
  'MED-2661': 402.0,
  'MED-118': 393.0,
  'MED-2651': 393.0,
  'MED-2658': 375.0,
  'MED-2652': 360.0,
  'MED-2495': 355.0,
  'MED-4505': 354.0,
  'MED-1271': 339.0},
 'PLAIN-33': {'MED-2721': 588.0,
  'MED-4763': 545.0,
  'MED-3058': 501.0,
  'MED-2717': 499.0,
  'MED-1803': 495.0,
  'MED-2722': 493.0,
  'MED-2723': 489.0,
  'MED-5006': 486.0,
  'MED-1468': 472.0,
  'MED-1797': 467.0},
 'PLAIN-44': {'MED-2811': 516.0,
  'MED-2817': 506.0,
  'MED-2831': 504.0,
  'MED-1948': 495.0,
  'MED-2814': 493.

In [18]:
### Evaluate the bm25
from evaluation.metrics import compute_ndcg

metrics = compute_ndcg(qrels, results) 
metrics_splade = compute_ndcg(qrels, splade_results)

In [19]:
metrics_splade

{'NDCG@1': 0.43344, 'NDCG@10': 0.31908, 'NDCG@100': 0.20898}

In [13]:
metrics

{'NDCG@1': 0.4257, 'NDCG@10': 0.32178, 'NDCG@100': 0.20768}

### Dense Retrieval

In [6]:
model_name = "BAAI/bge-base-en-v1.5"

In [ ]:
from dense.dense_retrievers import HNSW

hnsw = HNSW(model_name, f"{data_path}/corpus.jsonl")


In [7]:
hnsw.build_index()

In [34]:
import json

# Load the corpus to get doc_id mapping
doc_id_mapping = {}
with open(f"{data_path}/corpus.jsonl", "r") as f:
    for idx, line in enumerate(f):
        doc = json.loads(line)
        doc_id_mapping[idx] = doc['_id']  # Map index to document ID

# Now rebuild dense_results with correct doc IDs
dense_results = {}

for qid, query in queries.items():
    hits = hnsw.search([query])
    
    dense_results[qid] = {}
    for hit in hits[0]:
        # Convert numpy int to int, then map to actual doc ID
        doc_idx = int(hit['doc_id'])
        actual_doc_id = doc_id_mapping[doc_idx]
        dense_results[qid][actual_doc_id] = float(hit['score'])

print(dense_results)

{'PLAIN-2': {'MED-2429': 0.8610802888870239, 'MED-10': 0.8562972545623779, 'MED-14': 0.8529698848724365, 'MED-2431': 0.8503384590148926, 'MED-2428': 0.752158522605896, 'MED-2436': 0.7460639476776123, 'MED-4827': 0.7448838949203491, 'MED-2439': 0.7440789937973022, 'MED-5352': 0.7382776737213135, 'MED-3840': 0.7270352840423584}, 'PLAIN-12': {'MED-2512': 0.7218250632286072, 'MED-2325': 0.7194255590438843, 'MED-2510': 0.7140555381774902, 'MED-2502': 0.7135813236236572, 'MED-2514': 0.7106223702430725, 'MED-2501': 0.7105836868286133, 'MED-2520': 0.709314227104187, 'MED-2517': 0.7079290747642517, 'MED-4697': 0.7078589200973511, 'MED-4545': 0.697749674320221}, 'PLAIN-23': {'MED-2654': 0.7235015630722046, 'MED-118': 0.7211793065071106, 'MED-2651': 0.7211793065071106, 'MED-1961': 0.7111815214157104, 'MED-4726': 0.7110194563865662, 'MED-1164': 0.7075862288475037, 'MED-2658': 0.7045352458953857, 'MED-4168': 0.7031894326210022, 'MED-2494': 0.7011312246322632, 'MED-1169': 0.6971761584281921}, 'PLAIN

In [35]:
# Now evaluate
metrics = compute_ndcg(qrels, dense_results)
print(metrics)

{'NDCG@1': 0.45046, 'NDCG@10': 0.36304, 'NDCG@100': 0.23474}


## FlatIndex

In [8]:
from dense.dense_retrievers import FlatIndex

flat_index = FlatIndex(model_name, f"{data_path}/corpus.jsonl")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5683.30it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
flat_index.build_index()

In [10]:
import json

# Load the corpus to get doc_id mapping
doc_id_mapping_flat = {}
with open(f"{data_path}/corpus.jsonl", "r") as f:
    for idx, line in enumerate(f):
        doc = json.loads(line)
        doc_id_mapping_flat[idx] = doc['_id']  # Map index to document ID

# Now rebuild flatsearch_results with correct doc IDs
flatsearch_results = {}

for qid, query in queries.items():
    hits = flat_index.search([query])
    
    flatsearch_results[qid] = {}
    for hit in hits[0]:
        # Convert numpy int to int, then map to actual doc ID
        doc_idx = int(hit['doc_id'])
        actual_doc_id = doc_id_mapping_flat[doc_idx]
        flatsearch_results[qid][actual_doc_id] = float(hit['score'])

print(flatsearch_results)

{'PLAIN-2': {'MED-2429': 0.8610802888870239, 'MED-10': 0.8562972545623779, 'MED-14': 0.8529700040817261, 'MED-2431': 0.8503384590148926, 'MED-2428': 0.7521586418151855, 'MED-2436': 0.7460639476776123, 'MED-4827': 0.7448838949203491, 'MED-2439': 0.7440787553787231, 'MED-5352': 0.7382776737213135, 'MED-3840': 0.7270353436470032}, 'PLAIN-12': {'MED-2512': 0.7218250632286072, 'MED-2325': 0.7194255590438843, 'MED-2510': 0.7140555381774902, 'MED-2502': 0.7135813236236572, 'MED-2514': 0.7106223106384277, 'MED-2501': 0.7105836272239685, 'MED-2520': 0.709314227104187, 'MED-2517': 0.7079290747642517, 'MED-4697': 0.7078587412834167, 'MED-4545': 0.6977497935295105}, 'PLAIN-23': {'MED-2654': 0.7235015630722046, 'MED-2651': 0.7211793065071106, 'MED-118': 0.7211793065071106, 'MED-1961': 0.7111815214157104, 'MED-4726': 0.7110195159912109, 'MED-1164': 0.7075862288475037, 'MED-2658': 0.7045353055000305, 'MED-4168': 0.703189492225647, 'MED-2494': 0.7011312246322632, 'MED-1169': 0.6971760988235474}, 'PLAI

In [12]:
## Evaluate flatindex
from evaluation.metrics import compute_ndcg
metrics = compute_ndcg(qrels, flatsearch_results)
print(metrics)

{'NDCG@1': 0.45666, 'NDCG@10': 0.36574, 'NDCG@100': 0.23736}
